# TP1 : DU GSM (2G) À L'UMTS (3G) — CAPACITÉ CDMA, CELL BREATHING, SOFT HANDOVER
**Durée : 1h30** — Support Google Colab

### Objectifs
- Comprendre pourquoi un système CDMA fonctionne à C/I négatif (gain de traitement)
- Calculer la capacité d'une cellule UMTS (facteur de charge, capacité au pôle)
- Relier charge et couverture : la « respiration » de cellule (cell breathing)
- Chiffrer l'apport et le coût du soft handover
- Comparer avec le dimensionnement GSM du TP0 sur la même zone

### ⚠️ Instructions
Complétez les zones **`# TODO`**, exécutez dans l'ordre (Shift+Entrée), répondez aux questions **Réponse** en texte.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log10, sqrt, ceil
plt.rcParams['figure.figsize'] = (10, 5)
dB   = lambda x: 10*log10(x)          # linéaire -> dB
lin  = lambda x_dB: 10**(x_dB/10)     # dB -> linéaire
print("✅ Bibliothèques chargées")

---
## EXERCICE 1 : GAIN DE TRAITEMENT ET C/I EN CDMA (20 min)

> **Pour comprendre.** En GSM, chaque communication occupe un créneau de temps sur une fréquence de 200 kHz : deux utilisateurs
> ne se gênent pas parce qu'ils ne parlent jamais en même temps sur la même fréquence. En UMTS, tout le monde parle en même temps
> sur les mêmes 5 MHz. Ce qui sépare les utilisateurs, c'est un **code** : chaque bit utile est multiplié par une séquence
> de W/R « chips » (315 pour la voix). À la réception, le NodeB multiplie à nouveau par le même code : le signal utile se
> reconcentre (il est « désétalé ») alors que les autres utilisateurs, dont les codes sont différents, restent étalés et ne
> comptent que comme un bruit de fond. Ce mécanisme apporte un gain égal à W/R, appelé **gain de traitement** : c'est lui qui
> permet de décoder correctement un signal reçu 20 dB **en dessous** de l'interférence. Conséquence majeure : on n'a plus besoin
> d'éloigner les cellules qui utilisent la même fréquence, donc le motif K vaut 1.


En UMTS (WCDMA), tous les utilisateurs émettent **en même temps sur la même porteuse de 5 MHz**, séparés par des codes.
Le récepteur « désétale » le signal : le rapport signal sur interférence utile après désétalement est

$$rac{E_b}{N_0} = rac{W}{R}\cdotrac{C}{I} \qquad	ext{soit, en dB :}\qquad \left(rac{E_b}{N_0}ight)_{dB} = G_p + \left(rac{C}{I}ight)_{dB},\quad G_p = 10\log_{10}rac{W}{R}$$

$G_p$ est le **gain de traitement** (processing gain).

### Données
- Débit chip W = 3,84 Mcps — Voix AMR : R = 12,2 kbit/s, Eb/N0 requis = 5 dB
- Données : R = 384 kbit/s, Eb/N0 requis = 1,5 dB

In [ ]:
W = 3.84e6
services = {'Voix 12.2 kbit/s': (12.2e3, 5.0), 'Données 384 kbit/s': (384e3, 1.5)}

### Q1.1 — Gain de traitement G_p (linéaire et dB) pour chaque service

In [ ]:
for nom, (R, EbN0_dB) in services.items():
    Gp_lin = None  # TODO : COMPLÉTEZ ICI
    Gp_dB  = None  # TODO : COMPLÉTEZ ICI
    print(f"{nom:20s} : G_p = {Gp_lin:6.1f} = {Gp_dB:5.1f} dB")

### Q1.2 — C/I requis avant désétalement : C/I = Eb/N0 − G_p

In [ ]:
for nom, (R, EbN0_dB) in services.items():
    CI_dB = None  # TODO : COMPLÉTEZ ICI
    print(f"{nom:20s} : C/I requis = {CI_dB:6.1f} dB   (signal {abs(CI_dB):.0f} dB SOUS l'interférence !)")

**Réponse Q1.2** — Comparez avec le C/I de 9 dB du GSM (TP0). Pourquoi un réseau CDMA peut-il réutiliser la même fréquence dans toutes les cellules (K = 1) ?
Que se passe-t-il pour le C/I requis quand le débit R augmente ?

_(à compléter)_

---
## EXERCICE 2 : CAPACITÉ D'UNE CELLULE UMTS — LIEN MONTANT (30 min)

> **Pour comprendre.** Puisque les utilisateurs se voient mutuellement comme du bruit, chaque nouvel appel augmente
> l'interférence subie par tous les autres. Le NodeB demande alors à chacun de monter un peu en puissance pour conserver son
> Eb/N0, ce qui augmente encore l'interférence… Le niveau total reçu, rapporté au bruit thermique, s'appelle la **montée de bruit**
> (noise rise). Tant que la cellule est peu chargée, elle croît doucement ; près de la **capacité au pôle** elle diverge :
> le système devient instable. La capacité d'une cellule CDMA n'est donc pas un nombre fixe de canaux comme en GSM, mais une
> **quantité d'interférence que l'on s'autorise** : les opérateurs fixent une charge cible (50 à 75 %) et un contrôle d'admission
> refuse les appels au-delà. Le facteur (1 + i) traduit le fait que les mobiles des cellules voisines, sur la même fréquence,
> ajoutent leur propre interférence.


Chaque utilisateur $j$ apporte au NodeB une charge élémentaire

$$L_j = rac{1}{1 + \dfrac{W}{R_j\,(E_b/N_0)_j\,
u_j}}$$

où $
u$ est le facteur d'activité (0,67 en voix avec les silences). Le **facteur de charge** du lien montant est

$$\eta_{UL} = (1+i)\sum_j L_j \qquad i = rac{	ext{interférence des autres cellules}}{	ext{interférence de la cellule}} pprox 0{,}65 	ext{ (macro-cellule)}$$

La **montée de bruit** (noise rise) vue par le récepteur vaut

$$NR_{dB} = -10\log_{10}\left(1-\eta_{UL}ight)$$

Quand $\eta_{UL}	o 1$, $NR	o\infty$ : c'est la **capacité au pôle**.

### Données : voix 12,2 kbit/s, Eb/N0 = 5 dB, ν = 0,67, i = 0,65

In [ ]:
R_v, EbN0_v, nu, i_ratio = 12.2e3, 5.0, 0.67, 0.65

### Q2.1 — Charge élémentaire d'un utilisateur voix

In [ ]:
L_voix = None  # TODO : COMPLÉTEZ ICI
print(f"L = {L_voix:.4f}  → un utilisateur voix « consomme » {100*L_voix:.2f} % de la cellule (hors autres cellules)")

### Q2.2 — Facteur de charge pour N utilisateurs, et capacité au pôle N_pole (η_UL = 1)

In [ ]:
def charge_UL(N, L=L_voix, i=i_ratio):
    return None  # TODO : COMPLÉTEZ ICI

N_pole = None  # TODO : COMPLÉTEZ ICI
print(f"Capacité au pôle : N_pole = {N_pole:.1f} utilisateurs voix")
for N in (20, 40, 60):
    print(f"N = {N:3d} : η_UL = {charge_UL(N):.2f}")

### Q2.3 — Montée de bruit NR(η) et nombre d'utilisateurs admissibles à 50 % et 75 % de charge

In [ ]:
def noise_rise_dB(eta):
    return None  # TODO : COMPLÉTEZ ICI

for eta_cible in (0.5, 0.75):
    N_adm = None  # TODO : COMPLÉTEZ ICI
    print(f"η = {eta_cible:.0%} : NR = {noise_rise_dB(eta_cible):.1f} dB → {N_adm} utilisateurs voix")

### Graphique — montée de bruit en fonction du nombre d'utilisateurs

In [ ]:
N = np.arange(0, int(N_pole))
plt.plot(N, [noise_rise_dB(charge_UL(n)) for n in N], lw=2)
for eta_c, col in ((0.5, 'g'), (0.75, 'orange')):
    plt.axvline(eta_c*N_pole, color=col, ls='--', label=f'η = {eta_c:.0%} ({eta_c*N_pole:.0f} UE, NR = {noise_rise_dB(eta_c):.1f} dB)')
plt.axvline(N_pole, color='r', ls=':', label=f'capacité au pôle ({N_pole:.0f} UE)')
plt.ylim(0, 20); plt.xlabel('Utilisateurs voix simultanés'); plt.ylabel('Montée de bruit (dB)')
plt.title('Lien montant UMTS — noise rise'); plt.grid(alpha=.3); plt.legend(); plt.show()

**Réponse Q2.3** — Pourquoi les opérateurs planifient-ils à 50–75 % de charge et non à la capacité au pôle ?
Comment la capacité change-t-elle si i passe de 0,65 (macro) à 0,3 (cellule isolée) ? Refaites le calcul.

_(à compléter)_

---
## EXERCICE 3 : CELL BREATHING — LA COUVERTURE DÉPEND DE LA CHARGE (25 min)

> **Pour comprendre.** Un mobile en bordure de cellule émet déjà à sa puissance maximale (21 dBm). Si la cellule se charge,
> le bruit vu par le NodeB monte de NR décibels ; le mobile devrait émettre NR dB de plus pour rester audible, mais il ne peut pas :
> il est perdu. Autrement dit, la **portée réelle** de la cellule rétrécit quand elle se remplit et s'étend quand elle se vide,
> comme un poumon : c'est la respiration de cellule. En GSM, une cellule pleine refuse des appels (blocage) mais sa couverture
> ne bouge pas. En UMTS, couverture et capacité sont couplées, et le planificateur doit intégrer une **marge d'interférence**
> dans le bilan de liaison, égale à la montée de bruit correspondant à la charge cible.


Le bilan de liaison montant doit inclure une **marge d'interférence égale à la montée de bruit** : plus la cellule est chargée,
moins le mobile (puissance limitée) est entendu loin. Avec un affaiblissement $PL(r) = PL(1\,	ext{km}) + 10\,lpha\log_{10} r$, si le rayon à vide est $R_0$ :

$$R(\eta) = R_0\cdot 10^{-\dfrac{NR(\eta)}{10\,lpha}}$$

### Données
- Mobile : P_UE = 21 dBm, gain 0 dBi — NodeB : gain 17 dBi, pertes câble 3 dB, sensibilité à vide −120 dBm (voix)
- Marge de shadowing 8 dB — α = 3,5 — Affaiblissement de référence à 1 km : 130 dB (modèle COST-231 urbain)

In [ ]:
P_UE, G_UE, G_NB, L_cable, S_NB, M_shadow, alpha = 21, 0, 17, 3, -120, 8, 3.5
PL_1km = 130    # affaiblissement à 1 km (dB)

### Q3.1 — Affaiblissement maximal admissible à vide et rayon R_0

In [ ]:
PL_max_vide = None  # TODO : COMPLÉTEZ ICI
# PL(r) = PL_1km + 10·α·log10(r/1 km)  →  r = 10^((PL - PL_1km)/(10 α)) km
R_0 = None  # TODO : COMPLÉTEZ ICI
print(f"PL max à vide = {PL_max_vide:.1f} dB → R_0 = {R_0:.2f} km")

### Q3.2 — Rayon en charge : η = 25 %, 50 %, 75 %

In [ ]:
def rayon(eta):
    return None  # TODO : COMPLÉTEZ ICI

for eta in (0.25, 0.5, 0.75):
    print(f"η = {eta:.0%} : NR = {noise_rise_dB(eta):4.1f} dB → R = {rayon(eta):.2f} km  (surface {2.6*rayon(eta)**2:.2f} km²)")

### Graphique — respiration de cellule

In [ ]:
etas = np.linspace(0, 0.9, 50)
plt.plot(100*etas, [rayon(e) for e in etas], lw=2)
plt.xlabel('Charge de la cellule η (%)'); plt.ylabel('Rayon de couverture (km)')
plt.title('Cell breathing (lien montant, voix)'); plt.grid(alpha=.3); plt.show()

**Réponse Q3.2** — En GSM (TP0), le rayon dépendait-il de la charge ? Expliquez pourquoi, en UMTS, couverture et capacité ne peuvent plus être dimensionnées séparément.
Que risque-t-il de se passer pour un mobile en bordure quand la cellule voisine se charge brusquement ?

_(à compléter)_

---
## EXERCICE 4 : SOFT HANDOVER (15 min)

> **Pour comprendre.** Comme toutes les cellules utilisent la même fréquence, un mobile en bordure peut être reçu par deux ou
> trois NodeB en même temps. Plutôt que de basculer brutalement de l'un à l'autre (hard handover, GSM), le réseau combine les
> liens : c'est le **soft handover**. Il rend la bordure de cellule plus robuste (si un lien s'évanouit, l'autre tient) et
> apporte un gain de quelques dB sur le bilan de liaison. Mais il a un prix : chaque mobile en soft handover occupe une ressource
> (un code et de la puissance descendante) dans **chacune** des cellules concernées, ce qui augmente la charge du réseau
> d'environ un tiers. Le réglage de la fenêtre d'admission dans l'« active set » est le compromis entre robustesse et capacité.


En CDMA (K = 1), un mobile en bordure peut être écouté par **plusieurs NodeB simultanément** (soft handover) : le RNC combine
les deux liens. Effets :
- **gain de macro-diversité** sur le bilan montant : ≈ 2 à 3 dB (compense une partie de la marge de shadowing) ;
- **coût** : chaque mobile en SHO occupe un canal (code + puissance) dans chaque cellule de son active set. Avec une fraction f_SHO
  des mobiles en soft handover (typ. 30–40 %), la charge descendante et le nombre de canaux augmentent d'un facteur (1 + f_SHO).

In [ ]:
G_SHO_dB, f_SHO = 2.5, 0.35

### Q4.1 — Rayon à 50 % de charge avec le gain de soft handover

In [ ]:
R_50_sans = rayon(0.5)
R_50_avec = None  # TODO : COMPLÉTEZ ICI
print(f"η = 50 % : R = {R_50_sans:.2f} km sans SHO → {R_50_avec:.2f} km avec SHO  (surface +{100*((R_50_avec/R_50_sans)**2-1):.0f} %)")

### Q4.2 — Coût : canaux (codes) nécessaires pour servir N_adm utilisateurs à 50 % de charge

In [ ]:
N_adm_50 = int(0.5 * N_pole)
canaux_necessaires = None  # TODO : COMPLÉTEZ ICI
print(f"{N_adm_50} utilisateurs servis → {canaux_necessaires} canaux occupés (dont {canaux_necessaires - N_adm_50} pour le SHO)")

**Réponse Q4.2** — Le soft handover est-il possible en GSM ? Pourquoi ? Quel compromis l'opérateur règle-t-il en jouant sur la fenêtre d'admission dans l'active set ?

_(à compléter)_

---
## EXERCICE 5 : RETOUR SUR LA ZONE DU TP0 (10 min)

> **Pour comprendre.** L'intérêt du CDMA se voit en comparant deux réseaux sur la même zone. En GSM, le motif K = 7 imposait
> de ne donner à chaque cellule qu'un septième de la bande (17 canaux sur 125). En UMTS, chaque cellule dispose de la porteuse
> entière (K = 1) et sa capacité est limitée par l'interférence, pas par un plan de fréquences. Le résultat, avec 5 fois moins
> de spectre, est un réseau plusieurs fois moins dense. C'est le raisonnement qui a conduit la 3G, puis la 4G et la 5G, à
> abandonner la réutilisation de fréquences au profit de la gestion de l'interférence.


Même zone que le TP0 : **100 km², 100 000 abonnés, 25 mErlang/abonné** (2 500 Erlang). On déploie l'UMTS avec une porteuse de 5 MHz par cellule,
planifiée à 50 % de charge. On suppose qu'à 2 % de blocage, N canaux permettent d'écouler ≈ 0,8·N Erlang pour N ≈ 30 (table d'Erlang B).

In [ ]:
trafic_total = 2500
N_canaux_cell = int(0.5 * N_pole)                 # utilisateurs simultanés admissibles par cellule
erlang_par_cell = None  # TODO : COMPLÉTEZ ICI
cellules_capacite = None  # TODO : COMPLÉTEZ ICI
R_50 = rayon(0.5)
cellules_couverture = None  # TODO : COMPLÉTEZ ICI
print(f"Par cellule : {N_canaux_cell} canaux ≈ {erlang_par_cell:.1f} Erlang")
print(f"Cellules pour la capacité  : {cellules_capacite}")
print(f"Cellules pour la couverture (R à 50 % = {R_50:.2f} km) : {cellules_couverture}")
print(f"→ {max(cellules_capacite, cellules_couverture)} cellules, contre 234 cellules pour le GSM au TP0 (K = 7, 17 canaux/cellule)")

**Réponse Q5** — Le résultat vous surprend-il ? Où est passée la bande de 25 MHz du GSM ? Quel serait l'effet d'une deuxième porteuse UMTS ?

_(à compléter)_

---
## 📝 SYNTHÈSE — à remplir
| | Résultat |
|---|---|
| G_p voix / données | ___ dB / ___ dB |
| C/I requis voix | ___ dB |
| Capacité au pôle (voix, UL) | ___ utilisateurs |
| Utilisateurs à 50 % / 75 % | ___ / ___ |
| NR à 50 % / 75 % | ___ dB / ___ dB |
| R_0 / R(50 %) / R(75 %) | ___ / ___ / ___ km |
| Gain SHO sur R(50 %) | +___ % de surface |
| Cellules UMTS sur la zone du TP0 | ___ (GSM : 234) |

**Conclusion (5 lignes)** : en quoi le passage du TDMA/FDMA (GSM) au CDMA (UMTS) change-t-il la manière de planifier un réseau ?